In [0]:
catalog = "fraud_project"
schema_name = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_name}")

from pyspark.sql.functions import col, trim, lower, when, hour, to_timestamp, count, sum, avg, when, round, col
from pyspark.sql.types import DoubleType

ruta_fraude = "/Volumes/fraud_project/bronze/raw_data/input/csv/fraudTest.csv"

df_fraude = spark.read.option("header", True).option("inferSchema", True).csv(ruta_fraude)
display(df_fraude)

In [0]:
df_fraude.printSchema()

In [0]:
# Fact table-datos de cada transacción
df_fact_transacciones = df_fraude.select(
    col("trans_num"),
    col("cc_num"),
    col("merchant"),
    col("category"),
    col("amt"),
    col("trans_date_trans_time"),
    col("unix_time"),
    col("merch_lat"),
    col("merch_long"),
    col("is_fraud")
)
display(df_fact_transacciones)

In [0]:
# Dimensión cliente—un registro por cliente
df_dim_cliente = df_fraude.select(
    col("cc_num"),
    col("first"),
    col("last"),
    col("gender"),
    col("dob"),
    col("job"),
    col("street"),
    col("city"),
    col("state"),
    col("zip"),
    col("lat"),
    col("long"),
    col("city_pop")
).dropDuplicates(["cc_num"])

display(df_dim_cliente)

In [0]:
df_fact_clean = df_fact_transacciones \
    .withColumn("merchant", trim(lower(col("merchant")))) \
    .withColumn("category", trim(lower(col("category"))))
display(df_fact_clean)

In [0]:
# Columna calculada: es_madrugada
df_fact_clean = df_fact_clean \
    .withColumn("hora", hour(col("trans_date_trans_time"))) \
    .withColumn("es_madrugada", 
                when((col("hora") >= 0) & (col("hora") <= 5), True)
                .otherwise(False)) \
    .drop("hora")   # eliminamos hora

display(df_fact_clean)

In [0]:
# Join entre fact y dim
df_join = df_fact_clean.alias("f") \
    .join(
        df_dim_cliente.alias("d"),
        col("f.cc_num") == col("d.cc_num"),
        "left"
    ) \
    .select(
        col("f.trans_num").alias("id_transaccion"),
        col("f.cc_num"),
        col("d.first").alias("nombre"),
        col("d.last").alias("apellido"),
        col("d.gender").alias("genero"),
        col("d.city").alias("ciudad"),
        col("d.state").alias("estado"),
        col("d.job").alias("ocupacion"),
        col("f.merchant"),
        col("f.category"),
        col("f.amt").alias("monto"),
        col("f.trans_date_trans_time").alias("fecha_transaccion"),
        col("f.es_madrugada"),
        col("f.is_fraud").alias("fraude")
    ).dropDuplicates(["id_transaccion"])

display(df_join)

In [0]:
# Nota: No se incluyeron todas las columnas para mantener la tabla limpia y enfocada.
# Solo se seleccionaron las columnas más relevantes para análisis de fraude:
# - Datos del cliente (nombre, ciudad, ocupación)
# - Datos de la transacción (comercio, categoría, monto, fecha, madrugada, fraude)
# Columnas geográficas detalladas como lat, long, zip, street y dob se omitieron 
# porque no son prioritarias para esta etapa.

In [0]:
# Agregación 1: Fraudes por categoría de comercio
df_fraude_categoria = (
    df_join
    .groupBy("category")
    .agg(
        count("*").alias("total_transacciones"),
        sum(when(col("fraude") == 1, 1).otherwise(0)).alias("num_fraudes"),
        round(avg("monto"), 2).alias("monto_promedio"),
        round(sum("monto"), 2).alias("monto_total")
    )
)

In [0]:
display(df_fraude_categoria)

In [0]:
# Agregación 2: Fraudes por ciudad
df_fraude_ciudad = (
    df_join
    .groupBy("ciudad")
    .agg(
        count("*").alias("total_transacciones"),
        sum(when(col("fraude") == 1, 1).otherwise(0)).alias("num_fraudes"),
        round(avg("monto"), 2).alias("monto_promedio")
    )
)

In [0]:
display(df_fraude_ciudad)

In [0]:
# Agregación 3: Fraudes por madrugada
df_fraude_madrugada = (
    df_join
    .groupBy("es_madrugada")
    .agg(
        count("*").alias("total_transacciones"),
        sum(when(col("fraude") == 1, 1).otherwise(0)).alias("num_fraudes"),
        round(avg("monto"), 2).alias("monto_promedio")
    )
)

In [0]:
display(df_fraude_madrugada)

In [0]:
#Nombres de las tablas
tbl_detalle = f"{catalog}.{schema_name}.transacciones_detalle"
tbl_categoria = f"{catalog}.{schema_name}.fraudes_por_categoria"
tbl_ciudad = f"{catalog}.{schema_name}.fraudes_por_ciudad"
tbl_madrugada = f"{catalog}.{schema_name}.fraudes_por_madrugada"

In [0]:
#Guardar las tablas en Delta
df_join.write.format("delta").mode("overwrite").saveAsTable(tbl_detalle)

df_fraude_categoria.write.format("delta").mode("overwrite").saveAsTable(tbl_categoria)
df_fraude_ciudad.write.format("delta").mode("overwrite").saveAsTable(tbl_ciudad)
df_fraude_madrugada.write.format("delta").mode("overwrite").saveAsTable(tbl_madrugada)

In [0]:
%sql
SELECT * FROM fraud_project.silver.transacciones_detalle 
LIMIT 100;

In [0]:
%sql
SELECT * 
FROM fraud_project.silver.transacciones_detalle 
WHERE fraude = 1;